In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
from collections import deque

# Environment
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# DQN Model
model = nn.Sequential(
    nn.Linear(state_dim, 64),
    nn.ReLU(),
    nn.Linear(64, action_dim)
)

target = nn.Sequential(
    nn.Linear(state_dim, 64),
    nn.ReLU(),
    nn.Linear(64, action_dim)
)
target.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=0.001)
memory = deque(maxlen=1000)

# Collect one experience
state, _ = env.reset()
action = env.action_space.sample()
next_state, reward, done, trunc, _ = env.step(action)
done = done or trunc

memory.append((state, action, reward, next_state, done))

# Train using one sample
s, a, r, ns, d = random.choice(memory)

s = torch.FloatTensor(s)
ns = torch.FloatTensor(ns)

pred = model(s)[a]

with torch.no_grad():
    target_q = r + 0.99 * target(ns).max() * (1 - int(d))

loss = nn.MSELoss()(pred, target_q)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("DQN trained successfully!")

DQN trained successfully!
